In [31]:
import json
import pandas as pd

airports_df = pd.read_csv("../Output/brandon_airports_info.csv")

airports_df["iata_code"] = (
    airports_df["iata_code"]
    .astype(str)
    .str.strip()
    .str.upper()
)

valid_iata_codes = set(
    airports_df["iata_code"].dropna()
)

with open("../T100_Domestic_Market_and_Segment_Data_1289012864022719634.geojson", "r") as f:
    data = json.load(f)

travelers_df = pd.json_normalize(
    [feature["properties"] for feature in data["features"]]
)

travelers_df = travelers_df.drop(columns=['enplanements','freight','year','OBJECTID','mail']) # Drops irrelevant fields

travelers_df = travelers_df[
    travelers_df['origin'].isin(airports_df['iata_code']) # Only keep records that have IATA Code in the airports dataset
]

travelers_df = travelers_df.rename(columns = {"origin": "iata_code", "passengers": "approx_passengers", "departures": "approx_departures", "arrivals": "approx_arrivals"}  ) 

travelers_df.to_csv('../Output/travelers_cleaned.csv', index=False)

travelers_df.head(10)

,iata_code,approx_passengers,approx_departures,approx_arrivals
53,ABE,451490,6620,6488
54,ABI,97461,2315,2635
56,ABQ,2729021,30167,30040
57,ABR,23769,981,1024
58,ABY,30311,1276,1285
59,ACK,152003,10927,11083
60,ACT,57644,1175,1175
61,ACV,133676,2413,2629
62,ACY,499879,3659,3649
63,ADK,2179,123,123


In [30]:
import requests
import pandas as pd

all_holidays = []

# Request each year from the API
for year in range(2019, 2024):

    url = ("https://date.nager.at/api/v3/"f"PublicHolidays/{year}/US")
    response = requests.get(url)
    all_holidays.extend(response.json())

# Convert the API response into a DataFrame
holidays_df = pd.json_normalize(all_holidays)

holidays_df = holidays_df.drop(columns=['countryCode','fixed','global','counties','localName','launchYear'])

allowed_types = [["Public", "Bank"],["Observance"],["Bank"]]

holidays_df = holidays_df[holidays_df["types"].apply(lambda value: value in allowed_types)].reset_index(drop=True)
holidays_df = holidays_df.rename(columns = {"name": "holiday_name"}) 

holidays_df.to_csv('../Output/holidays_cleaned.csv', index=False)

holidays_df.head(10)

,date,holiday_name,types
0,2019-01-01,New Year's Day,"[Public, Bank]"
1,2019-01-21,"Martin Luther King, Jr. Day","[Public, Bank]"
2,2019-02-12,Lincoln's Birthday,[Observance]
3,2019-02-18,Presidents Day,"[Public, Bank]"
4,2019-05-27,Memorial Day,"[Public, Bank]"
5,2019-07-04,Independence Day,"[Public, Bank]"
6,2019-09-02,Labour Day,"[Public, Bank]"
7,2019-10-14,Columbus Day,[Bank]
8,2019-11-11,Veterans Day,"[Public, Bank]"
9,2019-11-28,Thanksgiving Day,"[Public, Bank]"
